# Genotype–Phenotype Heterogeneity and Heterogeneity-Driven Infertility Management in Non-Classical 21-Hydroxylase Deficiency: A Clinical Analysis and Literature Review Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and analyze the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

# Optional: display dataset citation, keywords, limitations, and collection details
print("\nDataset citation:", metadata.cite_as)
print("Keywords:", metadata.keywords)
print("Data limitations:", metadata.data_limitations)
print("Data collection timeframe:", metadata.data_collection_timeframe)
print("Data collection type:", metadata.data_collection_type)
print("Dataset version:", metadata.version)

## 2. Data Overview
Review available record sets, fields, and their IDs.
All entities are referenced by their `@id`.

We identify available record sets, their fields and columns, and print their structural details.

In [ ]:
# Retrieve all record sets and their IDs
record_sets = dataset.record_sets
print("Available record sets (by @id):")
record_set_ids = []
for rs in record_sets:
    print(f"  Record set name: {rs.name} (@id: {rs.id})")
    record_set_ids.append(rs.id)
    print("    Fields and columns:")
    for field in rs.fields:
        print(f"      Field: {field.name} (@id: {field.id}, dataType: {field.data_type})")
        if hasattr(field, 'column') and field.column is not None:
            print(f"        Column: {field.column.name} (@id: {field.column.id})")
        elif hasattr(field, 'columns') and field.columns is not None:
            for col in field.columns:
                print(f"        Column: {col.name} (@id: {col.id})")
    print("")
# Preview the first few records from each record set
for rs in record_sets:
    print(f"First 2 records from record set {rs.name} (@id: {rs.id}):")
    gen = dataset.records(record_set=rs.id)
    for ix, record in enumerate(gen):
        pprint.pprint(record)
        if ix >= 1:
            break

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.
Use the record set and field `@id`s from the overview.

We'll extract each data table using its `@id`. Typical record sets might be named Table 1, Table 2, Table 3.

In [ ]:
# Extract data from each record set (@id)
# List of record set @id values found above
dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"\nColumns for record set @id: {record_set_id}:")
    print(df.columns.tolist())
    print("Preview:")
    display(df.head())
# For demonstration, select the first record set for further analysis
active_record_set_id = record_set_ids[0]
active_df = dataframes[active_record_set_id]
print(f"\nSelected record set for EDA: {active_record_set_id}")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

Operations include removing outliers, transforming numeric distributions, and grouping by categorical attributes.

We'll select a numeric field (e.g., 'age_at_diagnosis') by its column `@id`, filter and normalize, and group by a category field (e.g., 'infertility').

In [ ]:
# Example: select 'age_at_diagnosis' and 'infertility' field by their @id

# Find potential numeric and category fields by inspecting record set structure
numeric_field_id = None
group_field_id = None
for rs in record_sets:
    if rs.id == active_record_set_id:
        for field in rs.fields:
            # For demonstration, first Integer or Float field is numeric_field; first Boolean/Text is group_field
            if numeric_field_id is None and field.data_type in ['Integer', 'Float', 'Number']:
                numeric_field_id = field.id
            if group_field_id is None and field.data_type in ['Boolean', 'Text'] and 'infertility' in field.name.lower():
                group_field_id = field.id

# Show the chosen field IDs
print(f"Numeric field for EDA: {numeric_field_id}")
print(f"Category/group field for EDA: {group_field_id}")

# EDA Filtering and Normalization
if numeric_field_id is not None and numeric_field_id in active_df.columns:
    # Set a threshold (example threshold: 20)
    threshold = 20
    filtered_df = active_df[active_df[numeric_field_id] > threshold]
    print(f"Filtered records where {numeric_field_id} > {threshold}:")
    display(filtered_df.head())

    # Normalize the numeric field
    mean_val = filtered_df[numeric_field_id].mean()
    std_val = filtered_df[numeric_field_id].std()
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - mean_val) / std_val
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by a categorical field if available
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped data by {group_field_id} (average {numeric_field_id}):")
        print(grouped_df.head())
else:
    print("No suitable numeric field found for EDA in the selected record set.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Below: a histogram of the numeric field normalized, and a bar chart for grouped means.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram for normalized numeric field
if numeric_field_id is not None and f"{numeric_field_id}_normalized" in filtered_df.columns:
    plt.figure(figsize=(6,4))
    sns.histplot(filtered_df[f"{numeric_field_id}_normalized"], kde=True, bins=10)
    plt.title(f"Distribution of Normalized {numeric_field_id}")
    plt.xlabel(f"{numeric_field_id}_normalized")
    plt.ylabel("Count")
    plt.show()

    # Barplot for grouped means
    if group_field_id and group_field_id in filtered_df.columns:
        plt.figure(figsize=(6,4))
        sns.barplot(
            x=grouped_df[group_field_id],
            y=grouped_df[numeric_field_id]
        )
        plt.title(f"Average {numeric_field_id} grouped by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.show()

## 6. Conclusion
This notebook demonstrated loading, overview, extraction, EDA, and visualization for the FAIR^2 clinical genotype–phenotype dataset using `mlcroissant`.

Key steps included referencing all entities by their `@id` for consistency, exploring record sets, extracting data, filtering and normalizing numeric features, and visualizing relationships. The dataset provides rich insight into rare disease genetics and phenotype analysis, but its small cohort size and clinical specificity require caution for generalization and model development.